# New Tokyo PoC Script

## 1 · Scene & Environment Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Use GPU 0, will use CPU otherwise
import numpy as np
import pandas as pd
import json
import csv as _csv
import poc_startup as poc
import core.rt as rt

path_suffix = "/home/polimi"

transmitters = [1, 2, 101, 102] # All transmitters
receivers = [1, 2, 101, 102]    # All receivers

# ── Ray-tracing scene ─────────────────────────────────────────────────────────
scene = poc.setup_scene(
    file_name=f"{path_suffix}/tokyo-polimi-dt-jsac/scenarios/ookayama_full_flat/ookayama.xml",
    frequency=60e9,
    bandwidth=1000e6,
)

poc.setup_antennas(
    transmitters=transmitters,
    receivers=receivers,
    pattern="panasonic_wigig_rsu",
    elevation_csv=f"{path_suffix}/tokyo-polimi-dt-jsac/panasonic-rsu-pattern/elevation_beam_16.csv",
    azimuth_csv=f"{path_suffix}/tokyo-polimi-dt-jsac/panasonic-rsu-pattern/azimuth_beam_16.csv",
    polarization="VH",
)

poc.configure_rt(
    verbose=False,
    time_checker=True,
    rt_diffraction=False,
    rt_corner_diffraction=False,
    rt_max_depth=4,
)

poc.configure_filters(
    transmitters=transmitters,
    receivers=receivers,
    use_kalman_filter=True,
    kalman_process_var=0.3,
    kalman_meas_var=0.8, # Lower = trust measurement more
    kalman_rt_var=3      # Higher = trust RT less
)

struc = poc.startup()

## 2 · Object configurator

In [ ]:
# ── Asset paths & TX powers ───────────────────────────────────────────────────
CAR_1_MESH       = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/coms.ply"
CAR_2_MESH       = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/estima.ply"
RSU_MESH         = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/rsu.ply"
TREE_MESH        = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/tree.ply"


poc.add_network_object(id=1, mesh_path=CAR_1_MESH, antenna_displacement=[0.096, 0.157001, 0.791505], tx_power_dbm=10)
poc.add_network_object(id=2, mesh_path=CAR_2_MESH, antenna_displacement=[-0.281679, 0.477371, 0.975445], tx_power_dbm=10)
poc.add_network_object(id=101, mesh_path=RSU_MESH, antenna_displacement=[0, 0, 5.0/2 + 0.5], tx_power_dbm=30)
poc.add_network_object(id=102, mesh_path=RSU_MESH, antenna_displacement=[0, 0, 5.0/2 + 0.5], tx_power_dbm=30)

trees = {}
trees[1] = [151.070, 137.283, 8.64/2]
trees[2] = [167.037, 135.432, 8.64/2]
trees[3] = [153.032, 149.437, 8.64/2]
trees[4] = [168.768, 147.313, 8.64/2]
trees[5] = [170.476, 159.486, 8.64/2]
trees[6] = [185.203, 157.361, 8.64/2]
trees[7] = [186.522, 160.437, 8.64/2]

for i in range(1, len(trees) + 1):
    poc.add_tree(id=i, mesh_path=TREE_MESH, position=trees[i])

## 3 · Showtime!

In [ ]:
import time
import json
import random
import numpy as np
import csv

udp_socket = struc["udp_socket"]
verbose = struc["verbose"]

def main():
    while True:
        # Receive data from the socket
        payload, address = udp_socket.recvfrom(1024*10)
        struc["latest_msg_address"] = address
        message = payload.decode()
        
        t_tot = time.time()
        # Parse the JSON message
        data = json.loads(message)

        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            msg_entries = data["data"]
        elif isinstance(data, list):
            msg_entries = data
        else:
            msg_entries = [data]

        type_val = None
        if len(msg_entries) > 0 and isinstance(msg_entries[0], dict):
            type_val = msg_entries[0].get("type")

        if isinstance(type_val, str) and "RT_CONFIGURATION" in type_val.upper():
            rt.manage_online_reconfiguration(msg_entries, struc, is_manual_override=False)
            continue

        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            data = data["data"]
        
        if verbose:
            print(" ")
            print("  - - - - - - - - - - - - - - - - -  NEW PREDICTION REQUEST  - - - - - - - - - - - - - - - - -  ")
            print(" ")

        # Extract current measured RSSI to calibrate the Digital Twin
        current_1vs2 = data[0]["data"]["RSSI_Vehicle1_2"] if data[0]["data"]["RSSI_Vehicle1_2"] != 0 else None
        current_1vs3 = data[0]["data"]["RSSI_Vehicle1_3"] if data[0]["data"]["RSSI_Vehicle1_3"] != 0 else None
        current_2vs3 = data[0]["data"]["RSSI_Vehicle2_3"] if data[0]["data"]["RSSI_Vehicle2_3"] != 0 else None
        
        if struc["use_kalman_filter"]:
            for key, filt in struc["filters"].items():
                filt.predict()
        

        # Update the locations in the scenario
        future_timestamp = data[1]["clock"]
        predicted_p1 = data[1]["data"]["x1"]
        predicted_p2 = data[1]["data"]["x2"]
        predicted_p3 = data[1]["data"]["x3"]
        t = time.time()

        rt.manage_location_message(f"LOC_UPDATE:veh2,{predicted_p2['x']},{predicted_p2['z']},{0.0},{predicted_p2['yaw']},0,0,0", struc)
        rt.manage_location_message(f"LOC_UPDATE:veh3,{predicted_p1['x']},{predicted_p1['z']},{0.0},{predicted_p1['yaw']},0,0,0", struc) # I know they are inverted!
        rt.manage_location_message(f"LOC_UPDATE:veh1,{predicted_p3['x']},{predicted_p3['z']},{0.0},{predicted_p3['yaw']},0,0,0", struc) # I know they are inverted!
    
        location_update_time = (time.time() - t) * 1000
        if struc["time_checker"]:
            print(f"        Location update took: {location_update_time:.2f} ms")

        # Compute filtered RSSI predictions
        t_c = time.time()
        raw_rssi_1vs2 = {}
        raw_rssi_1vs3 = {}
        raw_rssi_2vs3 = {}
        jit = struc["montecarlo_max_position_jitter"]
        
        is_monte_carlo_enabled = struc["montecarlo_realizations"] > 1 and struc["montecarlo_max_position_jitter"] > 0.0
        real = struc["montecarlo_realizations"] if is_monte_carlo_enabled else 1

        for i in range(real):
            if is_monte_carlo_enabled:
                struc["seed"] = random.randint(0, int(1e6))
            print(f"        RSSI Prediction iteration {i+1}/{real} with seed {struc['seed']}")
            manage_location_message(f"LOC_UPDATE:veh2,{predicted_p2['x'] + random.uniform(-jit, jit)},{predicted_p2['z'] + random.uniform(-jit, jit)},{0.0},{predicted_p2['yaw']},0,0,0", struc)
            raw_rssi_1vs2[i] = tx_power - get_path_loss("car_1", "car_2", struc) + correction - 2.5
            los_1_2 = get_los("car_1", "car_2", struc)
            raw_rssi_1vs3[i] = tx_power - get_path_loss("car_1", "car_3", struc) + correction + 7 + 1.80 - 0.25
            los_1_3 = get_los("car_1", "car_3", struc)
            raw_rssi_2vs3[i] = tx_power - get_path_loss("car_2", "car_3", struc) + correction
            los_2_3 = get_los("car_2", "car_3", struc)
            struc["rays_cache"] = {}  # Clear rays cache to force re-computation
            i += 1

        raw_rssi_1vs2 = sum(raw_rssi_1vs2.values()) / real
        raw_rssi_1vs3 = sum(raw_rssi_1vs3.values()) / real
        raw_rssi_2vs3 = sum(raw_rssi_2vs3.values()) / real

        if struc["use_kalman_filter"]:
            # Update Kalman filters
            rssi_1vs2 = struc["filters"][("1","2")].update(z_meas=current_1vs2, z_rt=raw_rssi_1vs2)
            rssi_1vs3 = struc["filters"][("1","3")].update(z_meas=current_1vs3, z_rt=raw_rssi_1vs3)
            rssi_2vs3 = struc["filters"][("2","3")].update(z_meas=current_2vs3, z_rt=raw_rssi_2vs3)
        
        if struc["use_adaptive_bias_filter"]:
            rssi_1vs2 = struc["filters"][("1","2")].step(predicted_rt=raw_rssi_1vs2, current_meas=current_1vs2)
            rssi_1vs3 = struc["filters"][("1","3")].step(predicted_rt=raw_rssi_1vs3, current_meas=current_1vs3)
            rssi_2vs3 = struc["filters"][("2","3")].step(predicted_rt=raw_rssi_2vs3, current_meas=current_2vs3)

        rssi_prediction_time = (time.time() - t_c) * 1000
        if struc["time_checker"]:
            print(f"        RT and RSSI extraction took: {rssi_prediction_time:.2f} ms")

        if struc["verbose"]:
            print(f"        Raw predicted RSSI:   1vs2={raw_rssi_1vs2:.2f} dBm, 1vs3={raw_rssi_1vs3:.2f} dBm, 2vs3={raw_rssi_2vs3:.2f} dBm")
            print(f"        Filtered RSSI:        1vs2={rssi_1vs2:.2f} dBm, 1vs3={rssi_1vs3:.2f} dBm, 2vs3={rssi_2vs3:.2f} dBm")

        # Prepare the response
        response = [{}]
        response[0]["clock"] = future_timestamp
        response[0]["data"] = {}
        response[0]["data"]["x1"] = predicted_p1
        response[0]["data"]["x2"] = predicted_p2
        response[0]["data"]["x3"] = predicted_p3
        response[0]["data"]["RSSI_Vehicle1_2"] = rssi_1vs2
        response[0]["data"]["RSSI_Vehicle1_3"] = rssi_1vs3
        response[0]["data"]["RSSI_Vehicle2_3"] = rssi_2vs3
        response[0]["data"]["raw_RSSI_Vehicle1_2"] = raw_rssi_1vs2
        response[0]["data"]["raw_RSSI_Vehicle1_3"] = raw_rssi_1vs3
        response[0]["data"]["raw_RSSI_Vehicle2_3"] = raw_rssi_2vs3

        response = json.dumps(response, default=lambda o: float(o) if isinstance(o, np.float32) else o)

        # Send back the response
        #udp_socket.sendto(response.encode(), ("20.0.0.10", 35944))
        udp_socket.sendto(response.encode(), address) # Use this for local testing script
        total_elapsed_time = (time.time() - t_tot) * 1000

        if verbose:
            print(" ")
            print(f"  - - - - - - - - - - - - - - - - -   Handled in {total_elapsed_time:.2f} ms   - - - - - - - - - - - - - - - - -  ")
            print(" ")
            print(" ")
        
        # Logging
        local_timestamp = time.time()
        with open(struc["log_file"], mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([local_timestamp, data[0]["clock"], future_timestamp,
                             message,
                             struc["sionna_location_db"][1]['x'], struc["sionna_location_db"][1]['y'], struc["sionna_location_db"][1]['angle'],
                             struc["sionna_location_db"][2]['x'], struc["sionna_location_db"][2]['y'], struc["sionna_location_db"][2]['angle'],
                             struc["sionna_location_db"][3]['x'], struc["sionna_location_db"][3]['y'], struc["sionna_location_db"][3]['angle'],
                             raw_rssi_1vs2, raw_rssi_1vs3, raw_rssi_2vs3, 
                             rssi_1vs2, rssi_1vs3, rssi_2vs3,
                             location_update_time, rssi_prediction_time, total_elapsed_time, los_1_2, los_1_3, los_2_3])
        
        '''
        from sionna.rt import render_to_file
        # Frame-by-frame exporter
        global frame_num
        print("Exporting frame ", frame_num)
        
        struc["scene"].render_to_file(camera=my_cam, 
                                         filename=f"/home/rpegurri/Tokyo NDT Integration/PoC Tokyo Science/export/frame_{frame_num}.png", 
                                         show_devices=False)
        frame_num += 1
        '''
        

        if message.startswith("SHUTDOWN_SIONNA"):
            print("Got SHUTDOWN_SIONNA message. Bye!")
            udp_socket.close()
            break

# Entry point
if __name__ == "__main__":
    main()